# VVRAG ingestion (free GPU) — sample.pdf + sample.png
Self-contained: run top to bottom, no setup needed. Clones the repo, installs deps, probes the environment (retriever Option A check), ingests `data/raw/sample.pdf` (text layer) + `data/raw/sample.png` (via Tesseract OCR) with real ColQwen embeddings, verifies the manifest, zips `data/index` for download. No logic lives here.

In [ ]:
%cd /content
!if [ -d altVRAG ]; then git -C altVRAG pull -q; else git clone https://github.com/BiprashPandey/altVVRAG.git altVRAG; fi
%cd /content/altVRAG
!pip install -q -e . --no-deps
!mkdir -p data/raw && ls -lh data/raw

## 0. Confirm the sample files (no upload needed)
`sample.pdf` and `sample.png` ship with the repo clone, so nothing to upload — just re-run the `ls` cell above and confirm both are listed before continuing. (To ingest your own documents instead, drop them into `altVRAG/data/raw/` via the Files panel; note `data/` is gitignored so uploads never affect the repo.)

In [ ]:
!nvidia-smi || echo 'NO GPU VISIBLE -> Runtime > Change runtime type > T4 GPU'
%pip install -q torch --index-url https://download.pytorch.org/whl/cu121
%pip install -q colpali-engine qdrant-client pymupdf pillow pytesseract transformers groq openai google-genai
!pip install -q -U torchao  # peft requires torchao for the ColQwen LoRA adapter
!apt-get install -y tesseract-ocr > /dev/null 2>&1
!tesseract --version | head -1  # sample.png has no text layer: OCR must work
!python -c "import torch; print('torch', torch.__version__, '| cuda build:', torch.version.cuda, '| available:', torch.cuda.is_available())"
!python -c "from vvrag.ingest.embed import probe_environment; probe_environment()"

## 1. Verify the retriever checkpoint (Option A)
Check that `vidore/colqwen2-v1.0` resolves on the Hub (metadata lookup, downloads nothing). If it fails (gated/renamed), log in (`!hf login`) or pick a working ColPali checkpoint and pass `--model-id` in cell 3.

In [ ]:
from huggingface_hub import HfApi
info = HfApi().model_info("vidore/colqwen2-v1.0")  # metadata only, no download
print("id:", info.id, "| private:", info.private, "| gated:", getattr(info, "gated", None))
# 401/GatedRepoError -> run `!hf login` first. 404 -> set MODEL_ID to a verified
# fallback (e.g. vidore/colpali-v1.2) and pass --model-id in the ingest cell.

## 2. Ingest sample.pdf + sample.png (real ColQwen embeddings)
Do NOT pass `--skip-ocr`: `sample.png` is an image page and gets its word boxes from Tesseract (installed above). `sample.pdf` already has a text layer (~166 words). Only 2 files, so no `--limit` smoke run — ingest everything in `data/raw`.

Name collision warning: ingest uses the file stem as `doc_id`, so `sample.pdf` and `sample.png` would both become doc `sample` page 0 (the png would be skipped as already-indexed). The ingest cell renames the png to `sample_page.png` first — docs end up as `sample` (pdf) + `sample_page` (png).

In [ ]:
!ls -lh data/raw
!test -f data/raw/sample.pdf || (echo 'MISSING data/raw/sample.pdf -> do step 0 first' && exit 1)
!if [ -f data/raw/sample.png ]; then mv data/raw/sample.png data/raw/sample_page.png && echo 'renamed sample.png -> sample_page.png (avoids doc_id collision with sample.pdf)'; fi
!test -f data/raw/sample_page.png || (echo 'MISSING data/raw/sample_page.png -> do step 0 first' && exit 1)
!python -m vvrag.ingest.run_ingest --input data/raw --out data/index --embedder colqwen
# Re-running is safe (resumable: finished pages are skipped).

## 3. Verify the index (sample + sample_page must be present)
Expect doc `sample` (pdf) and doc `sample_page` (png) each with word boxes, and `word_boxes` well above 166 (pdf text words + png OCR words). Note: the repo skeleton also ships `testimg.png`, so totals read docs/pages = 3 — that file is harmless extra. If the `sample_page` row is missing word boxes, Tesseract found nothing — check `tesseract --version` output in step 1.

In [ ]:
!cat data/index/manifest.json
!ls -lh data/index/embeddings/
import json, sqlite3
m = json.load(open('data/index/manifest.json'))
assert m['stats']['pages'] >= 2, m['stats']
con = sqlite3.connect('data/index/meta.db')
got = {(d, p): n for d, p, n in con.execute('SELECT doc_id, page, COUNT(*) FROM word_boxes GROUP BY doc_id, page')}
for (doc, page), n in sorted(got.items()):
    print(f'words: {doc}/{page}: {n}')
assert ('sample', 0) in got, got
assert ('sample_page', 0) in got, got
n = con.execute('SELECT COUNT(*) FROM word_boxes').fetchone()[0]
assert n > 100, n
print(f'OK: sample + sample_page indexed, {n} word boxes total')

In [ ]:
!zip -qr index.zip data/index && ls -lh index.zip
# Download index.zip (Files panel) and unzip it into data/index/ on your laptop.
# The zip is self-contained: pages, embeddings, meta.db, manifest AND the qdrant index.